# ATML PA3 Task 2: Kaggle Runner
Selects the data, caches features, trains the 4,500-update curriculum, runs the final evaluation, then executes all seven notebooks **with their outputs**.

- **Accelerator:** GPU T4 x2 (the run uses one GPU, or both with the optional 49-token run). **Internet:** ON.
- **Run with:** Save Version -> Save & Run All.
- **Smoke test first:** set `SMOKE_TEST = True` and run once. It tests training, resuming, and evaluation on the real models in minutes, writing only to `/tmp`.
- **Multiple sessions:** training stops itself near the time limit. To continue, Add Input -> Your Work -> the previous version, then run again.
- **Afterwards:** download `task2_outputs.zip` and unzip it into `task2_vlm_grounding/`. Checkpoints and features stay out of git.

In [ ]:
import glob, os, shutil, subprocess, threading, time
SMOKE_TEST = True               # True: quick test on real models. False: the real experiment.
RUN_COMPRESSION = False         # Optional 49-token experiment on the second GPU
BRANCH = "fix/task1-pipeline"   # Change to "master" once the branch is merged
SESSION_HOURS = 11.0
SESSION_START = time.time()

CODE = "/tmp/ATML-PA3"
OUT = "/kaggle/working/task2_outputs"
RAW, IMAGES = "/tmp/task2_raw", "/tmp/task2_images"
FEATURES = f"{OUT}/features"
MANIFEST = f"{OUT}/splits/manifest_seed6304.json"
NOTEBOOKS = ["1_siglip_feature_caching", "2_data_preparation_and_curriculum", "3_stage1_caption_alignment", "4_stage2_visual_instruction_tuning",
             "5_stage3_vqa_specialization", "6_grounding_evaluation_and_controls", "7_token_compression_ablation"]

def minutes_left():
    return SESSION_HOURS * 60 - (time.time() - SESSION_START) / 60

def run(command, gpu=None, label=""):
    env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    process = subprocess.Popen(command, shell=True, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print((f"[{label}] " if label else "") + line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Failed: {command}")

shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/moizejaz-31/ATML-PA3.git", CODE], check=True)
run('pip install -q -U "transformers>=4.51" accelerate seaborn pyyaml nbconvert ipykernel')

previous = sorted(glob.glob("/kaggle/input/**/task2_outputs", recursive=True))
if previous:
    print("Restoring", previous[-1])
    shutil.copytree(previous[-1], OUT, dirs_exist_ok=True)
for folder in ["results/main", "splits", "notebooks", "checkpoints"]:
    os.makedirs(f"{OUT}/{folder}", exist_ok=True)

In [ ]:
# Select data, download only the selected images, cache SigLIP features
run(f"python -m task2_vlm_grounding.src.data --raw_dir {RAW} --manifest {MANIFEST} --image_dir {IMAGES}")
run(f"python -m task2_vlm_grounding.src.feature_cache --manifest {MANIFEST} --image_dir {IMAGES} --output_dir {FEATURES}", gpu=0)

In [ ]:
# Smoke test: real models, 6 + 3 + 3 updates, stop and resume, then the final evaluation
if SMOKE_TEST:
    smoke = "/tmp/task2_smoke"
    shutil.rmtree(smoke, ignore_errors=True)
    start = time.time()
    train = f"python -m task2_vlm_grounding.src.train --manifest {MANIFEST} --feature_dir {FEATURES} --run_dir {smoke}/ckpt --results_dir {smoke}/results/main --stage_updates 6,3,3 --save_every 2 --eval_every 6"
    run(f"{train} --until_update 4", gpu=0, label="stop")
    run(train, gpu=0, label="resume")
    run(f"python -m task2_vlm_grounding.src.train --manifest {MANIFEST} --feature_dir {FEATURES} --run_dir {smoke}/ckpt --results_dir {smoke}/results/main --final_only", gpu=0, label="final")
    print(open(f"{smoke}/results/main/final_summary.csv").read()[:1200])
    shutil.rmtree(smoke)
    print(f"\nSMOKE TEST PASSED in {(time.time() - start) / 60:.1f} min. Set SMOKE_TEST = False for the real run.")

In [ ]:
# Training. Stops itself before the session limit and resumes next session.
if not SMOKE_TEST:
    reserve = 75
    limit = minutes_left() - reserve
    def train(run_name, tokens, gpu):
        run(f"python -m task2_vlm_grounding.src.train --manifest {MANIFEST} --feature_dir {FEATURES} --run_dir {OUT}/checkpoints/{run_name} --results_dir {OUT}/results/{run_name} --visual_tokens {tokens} --time_limit_min {limit:.0f}", gpu=gpu, label=run_name)
    threads = [threading.Thread(target=train, args=("main", 196, 0))]
    if RUN_COMPRESSION:
        threads.append(threading.Thread(target=train, args=("compressed_49", 49, 1)))
    for t in threads:
        t.start()
    for t in threads:
        t.join()

In [ ]:
# Final evaluation of each finished run, then execute the notebooks
if not SMOKE_TEST:
    import json
    finished = []
    for run_name, tokens in [("main", 196), ("compressed_49", 49)]:
        path = f"{OUT}/results/{run_name}/summary.json"
        if os.path.exists(path) and json.load(open(path))["completed"]:
            finished.append(run_name)
            if not os.path.exists(f"{OUT}/results/{run_name}/final_summary.csv"):
                run(f"python -m task2_vlm_grounding.src.train --manifest {MANIFEST} --feature_dir {FEATURES} --run_dir {OUT}/checkpoints/{run_name} --results_dir {OUT}/results/{run_name} --visual_tokens {tokens} --final_only", gpu=0, label=f"final {run_name}")
    print("Finished runs:", finished or "none yet; attach this version and run again")

    env = dict(os.environ, TASK2_OUTPUT_DIR=OUT, TASK2_MANIFEST=MANIFEST, TASK2_IMAGE_DIR=IMAGES, TASK2_FEATURE_DIR=FEATURES, TASK2_RESULTS_DIR=f"{OUT}/results")
    if "main" in finished:
        for name in NOTEBOOKS:
            result = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3",
                                     "--output", f"{name}.ipynb", "--output-dir", f"{OUT}/notebooks", f"{CODE}/task2_vlm_grounding/notebooks/{name}.ipynb"], env=env, capture_output=True, text=True)
            if result.returncode != 0:
                print(result.stdout[-3000:], result.stderr[-6000:])
                raise RuntimeError(f"{name} failed")
            print("Executed", name, flush=True)

In [ ]:
# Package results to download. Features and checkpoints stay in the session output only.
if not SMOKE_TEST:
    subprocess.run(f"rm -f /kaggle/working/task2_outputs.zip && cd {OUT} && zip -q -r /kaggle/working/task2_outputs.zip results splits notebooks", shell=True, check=True)
    run(f"du -sh {OUT} {OUT}/*")
    from IPython.display import Image, display
    for path in sorted(glob.glob(f"{OUT}/results/figures/*.png")):
        print(os.path.basename(path))
        display(Image(filename=path, width=900))